# Assignment 4: Named Entity Recognition — Broad Twitter Corpus
**Dataset:** GateNLP/broad_twitter_corpus

**Entities:** PERSON, LOCATION, ORGANIZATION, PRODUCT  

**Model:** dslim/bert-base-NER  

## 0. Install Dependencies

In [1]:
!pip install transformers torch seqeval scikit-learn pandas pyarrow

## 1. Load the Broad Twitter Corpus via Parquet

In [2]:
# NOTE: If the above cell fails with a 404, run this cell instead.
# It fetches the exact Parquet URLs from the API and loads them.

import requests, pandas as pd
from io import BytesIO

def load_split_from_hf(dataset, config, split):
    api = f"https://huggingface.co/api/datasets/{dataset}/parquet/{config}/{split}"
    resp = requests.get(api)
    urls = resp.json()  # list of parquet file URLs
    print(f"Found {len(urls)} parquet file(s) for {split}:", urls)
    dfs = []
    for url in urls:
        r = requests.get(url)
        dfs.append(pd.read_parquet(BytesIO(r.content)))
    return pd.concat(dfs, ignore_index=True)

train_df = load_split_from_hf('GateNLP/broad_twitter_corpus', 'broad-twitter-corpus', 'train')
test_df  = load_split_from_hf('GateNLP/broad_twitter_corpus', 'broad-twitter-corpus', 'test')

print(f"\nTrain size: {len(train_df)}")
print(f"Test size:  {len(test_df)}")
print("\nColumns:", train_df.columns.tolist())
print("\nSample row:")
print(train_df.iloc[0])

Found 1 parquet file(s) for train: ['https://huggingface.co/api/datasets/GateNLP/broad_twitter_corpus/parquet/broad-twitter-corpus/train/0.parquet']
Found 1 parquet file(s) for test: ['https://huggingface.co/api/datasets/GateNLP/broad_twitter_corpus/parquet/broad-twitter-corpus/test/0.parquet']

Train size: 5342
Test size:  2002

Columns: ['id', 'tokens', 'ner_tags']

Sample row:
id                                                          0
tokens      [I, hate, the, words, chunder, ,, vomit, and, ...
ner_tags                 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Name: 0, dtype: object


## 1b. Clean Dataset — Remove Non-English & Emoji Tweets

In [3]:
import re
import unicodedata

# ================================================================
# 1b. Filter: Remove non-English tweets and tweets with emojis
# ================================================================

def is_clean_english(tokens):
    text = ' '.join(tokens)

    # Reject if any character is an emoji or special unicode symbol
    for char in text:
        cat = unicodedata.category(char)
        if cat in ('So', 'Cs', 'Co'):  # Symbol-other, surrogates, private use
            return False
        cp = ord(char)
        if (0x1F300 <= cp <= 0x1FFFF) or (0x2600 <= cp <= 0x27BF):
            return False

    # Reject if text contains non-ASCII characters (catches non-English)
    try:
        text.encode('ascii')
    except UnicodeEncodeError:
        return False

    # Reject if too short to be useful for NER
    if len(tokens) < 4:
        return False

    # Reject if mostly punctuation/symbols with barely any real words
    alpha_tokens = [t for t in tokens if re.search(r'[a-zA-Z]', t)]
    if len(alpha_tokens) / len(tokens) < 0.4:
        return False

    return True

before_train = len(train_df)
before_test  = len(test_df)

train_df['keep'] = train_df['tokens'].apply(is_clean_english)
test_df['keep']  = test_df['tokens'].apply(is_clean_english)

train_df = train_df[train_df['keep']].drop(columns='keep').reset_index(drop=True)
test_df  = test_df[test_df['keep']].drop(columns='keep').reset_index(drop=True)

print(f"Train: {before_train} → {len(train_df)} docs kept ({before_train - len(train_df)} removed)")
print(f"Test:  {before_test} → {len(test_df)} docs kept ({before_test - len(test_df)} removed)")

print("\nSample cleaned tweets:")
for i in range(5):
    print(f"  [{i}] {' '.join(train_df.iloc[i]['tokens'])}")


Train: 5342 → 4836 docs kept (506 removed)
Test:  2002 → 1688 docs kept (314 removed)

Sample cleaned tweets:
  [0] I hate the words chunder , vomit and puke . BUUH .
  [1] Alesan kenapa mlm kita lbh srg galau Poconggg '"' TwitFAKTA : Otak lebih aktif di malam hari dari pada di pagi hari . # TwitFAKTA '"'
  [2] Complete Tosca on the tube http://t.co/O90deSLB
  [3] Think you call that smash and grab . # Gateshead 's media man just admitted to me it was '"' daylight robbery '"' . Shaw 's only touch was his goal .
  [4] Happy New Year world !


In [4]:
# Discover the label mapping
# Broad Twitter Corpus standard labels:
label_list = [
    'O',
    'B-PER', 'I-PER',
    'B-LOC', 'I-LOC',
    'B-ORG', 'I-ORG',
    'B-PROD', 'I-PROD'
]

# Check what the ner_tags column looks like
print("Sample ner_tags:", train_df['ner_tags'].iloc[0])
print("Sample tokens:",   train_df['tokens'].iloc[0])

# Find max tag ID to verify label list covers everything
all_tag_ids = [t for tags in train_df['ner_tags'] for t in tags]
print(f"\nMax tag ID in data: {max(all_tag_ids)}")
print(f"Label list length:  {len(label_list)}")

print("\nLabel mapping:")
for i, label in enumerate(label_list):
    print(f"  {i}: {label}")

Sample ner_tags: [0 0 0 0 0 0 0 0 0 0 0 0]
Sample tokens: ['I' 'hate' 'the' 'words' 'chunder' ',' 'vomit' 'and' 'puke' '.' 'BUUH'
 '.']

Max tag ID in data: 6
Label list length:  9

Label mapping:
  0: O
  1: B-PER
  2: I-PER
  3: B-LOC
  4: I-LOC
  5: B-ORG
  6: I-ORG
  7: B-PROD
  8: I-PROD


In [5]:
# Helper functions
def tokens_to_text(tokens):
    return ' '.join(tokens)

# Show first 5 tweets with their entity labels
print("=== FIRST 5 TWEETS ===")
for i in range(5):
    row = train_df.iloc[i]
    text = tokens_to_text(row['tokens'])
    tags = [label_list[t] for t in row['ner_tags']]
    entities = [(row['tokens'][j], tags[j]) for j in range(len(tags)) if tags[j] != 'O']
    print(f"\n[{i}] {text}")
    print(f"     Entities: {entities}")

=== FIRST 5 TWEETS ===

[0] I hate the words chunder , vomit and puke . BUUH .
     Entities: []

[1] Alesan kenapa mlm kita lbh srg galau Poconggg '"' TwitFAKTA : Otak lebih aktif di malam hari dari pada di pagi hari . # TwitFAKTA '"'
     Entities: []

[2] Complete Tosca on the tube http://t.co/O90deSLB
     Entities: []

[3] Think you call that smash and grab . # Gateshead 's media man just admitted to me it was '"' daylight robbery '"' . Shaw 's only touch was his goal .
     Entities: [('Gateshead', 'B-ORG'), ('Shaw', 'B-PER')]

[4] Happy New Year world !
     Entities: []


## 2. Apply Pretrained NER Model (dslim/bert-base-NER)

In [6]:
from transformers import pipeline

ner_pipeline = pipeline(
    'ner',
    model='dslim/bert-base-NER',
    aggregation_strategy='simple'
)
print("Model loaded: dslim/bert-base-NER")
print("Labels: PER, LOC, ORG, MISC")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: dslim/bert-base-NER
Labels: PER, LOC, ORG, MISC


In [7]:
SAMPLE_SIZE = 200
sentences = [tokens_to_text(train_df.iloc[i]['tokens']) for i in range(SAMPLE_SIZE)]

print(f"Running NER on {SAMPLE_SIZE} tweets...")
predictions = ner_pipeline(sentences, batch_size=16)
print("Done.")

Running NER on 200 tweets...
Done.


In [8]:
# Display predictions — screenshot this section for your checkpoint report!
print("=== SAMPLE PREDICTIONS (first 15) ===\n")
for i in range(15):
    row = train_df.iloc[i]
    gold_tags    = [label_list[t] for t in row['ner_tags']]
    gold_entities = [(row['tokens'][j], gold_tags[j]) for j in range(len(gold_tags)) if gold_tags[j] != 'O']

    print(f"[{i}] {sentences[i]}")
    print(f"  GOLD:      {gold_entities}")
    if predictions[i]:
        for ent in predictions[i]:
            print(f"  PREDICTED: [{ent['entity_group']}] '{ent['word']}' (score: {ent['score']:.2f})")
    else:
        print("  PREDICTED: (no entities found)")
    print()

=== SAMPLE PREDICTIONS (first 15) ===

[0] I hate the words chunder , vomit and puke . BUUH .
  GOLD:      []
  PREDICTED: (no entities found)

[1] Alesan kenapa mlm kita lbh srg galau Poconggg '"' TwitFAKTA : Otak lebih aktif di malam hari dari pada di pagi hari . # TwitFAKTA '"'
  GOLD:      []
  PREDICTED: (no entities found)

[2] Complete Tosca on the tube http://t.co/O90deSLB
  GOLD:      []
  PREDICTED: [MISC] 'Tosca' (score: 0.95)

[3] Think you call that smash and grab . # Gateshead 's media man just admitted to me it was '"' daylight robbery '"' . Shaw 's only touch was his goal .
  GOLD:      [('Gateshead', 'B-ORG'), ('Shaw', 'B-PER')]
  PREDICTED: [LOC] 'Gateshead' (score: 0.92)
  PREDICTED: [PER] 'Shaw' (score: 1.00)

[4] Happy New Year world !
  GOLD:      []
  PREDICTED: (no entities found)

[5] middle aged man band playing blink 182 . l0 l .
  GOLD:      [('blink', 'B-LOC'), ('182', 'I-LOC'), ('.', 'I-LOC')]
  PREDICTED: (no entities found)

[6] 14 days till my birthday 

## 3. Error Analysis (FP, FN, Label Mismatches)
**For your report:** The model uses general labels (PER, LOC, ORG, MISC). Twitter entities like informal name mentions, slang locations, and brand names are hard for a news-trained model — document these failures.

In [9]:
def get_gold_entities(row, label_list):
    entities = []
    tokens = row['tokens']
    tags   = row['ner_tags']
    i = 0
    while i < len(tags):
        tag = label_list[tags[i]]
        if tag.startswith('B-'):
            label = tag[2:]
            start = i
            i += 1
            while i < len(tags) and label_list[tags[i]] == f'I-{label}':
                i += 1
            span_text = ' '.join(tokens[start:i])
            entities.append({'text': span_text, 'label': label})
        else:
            i += 1
    return entities

gold_entities_all = [get_gold_entities(train_df.iloc[i], label_list) for i in range(SAMPLE_SIZE)]

In [10]:
from collections import Counter

false_positives = []
false_negatives = []
span_matches    = []

for i in range(SAMPLE_SIZE):
    gold = {e['text'].lower(): e['label'] for e in gold_entities_all[i]}
    pred = {e['word'].lower(): e['entity_group'] for e in predictions[i]}

    for text, label in pred.items():
        if text not in gold:
            false_positives.append({'tweet': sentences[i], 'text': text, 'pred_label': label})
        else:
            span_matches.append({'tweet': sentences[i], 'text': text,
                                  'gold_label': gold[text], 'pred_label': label})

    for text, label in gold.items():
        if text not in pred:
            false_negatives.append({'tweet': sentences[i], 'text': text, 'gold_label': label})

print(f"False Positives (predicted, not in gold): {len(false_positives)}")
print(f"False Negatives (in gold, model missed):  {len(false_negatives)}")
print(f"Span matches (found, label differs):      {len(span_matches)}")

False Positives (predicted, not in gold): 121
False Negatives (in gold, model missed):  49
Span matches (found, label differs):      43


In [11]:
print("\n=== FALSE POSITIVES (first 5) ===")
for e in false_positives[:5]:
    print(f"  Tweet: {e['tweet']}")
    print(f"  Predicted '{e['text']}' as {e['pred_label']} — not in gold")
    print()

print("\n=== FALSE NEGATIVES (first 5) ===")
for e in false_negatives[:5]:
    print(f"  Tweet: {e['tweet']}")
    print(f"  Missed '{e['text']}' ({e['gold_label']})")
    print()

print("\n=== LABEL MISMATCHES (first 5) ===")
for e in span_matches[:5]:
    print(f"  Tweet: {e['tweet']}")
    print(f"  '{e['text']}' | Gold: {e['gold_label']} | Pred: {e['pred_label']}")
    print()


=== FALSE POSITIVES (first 5) ===
  Tweet: Complete Tosca on the tube http://t.co/O90deSLB
  Predicted 'tosca' as MISC — not in gold

  Tweet: Happy New Year 2012 # fireworks # HappyNewYear # photography http://t.co/ImHeLldm
  Predicted '##ew' as MISC — not in gold

  Tweet: How did Dorothy Gale come back * younger * in Return to Oz ? ?
  Predicted 'return to oz' as MISC — not in gold

  Tweet: Check out the new # TantamountToTreaser preview on our website ! http://t.co/NiqVnls3
  Predicted 'tan' as MISC — not in gold

  Tweet: TD Chargers = \ . If we ca n't beat the Chargers we do n't deserve to go to the playoffs .
  Predicted 'chargers' as ORG — not in gold


=== FALSE NEGATIVES (first 5) ===
  Tweet: middle aged man band playing blink 182 . l0 l .
  Missed 'blink 182 .' (LOC)

  Tweet: TD Chargers = \ . If we ca n't beat the Chargers we do n't deserve to go to the playoffs .
  Missed 'td chargers' (LOC)

  Tweet: TD Chargers = \ . If we ca n't beat the Chargers we do n't deserve t

In [12]:
fn_labels = Counter(e['gold_label'] for e in false_negatives)
fp_labels = Counter(e['pred_label'] for e in false_positives)

print("Most commonly MISSED entity types:")
for label, count in fn_labels.most_common():
    print(f"  {label}: {count}")

print("\nMost common spurious predictions:")
for label, count in fp_labels.most_common():
    print(f"  {label}: {count}")

Most commonly MISSED entity types:
  PER: 23
  LOC: 15
  ORG: 11

Most common spurious predictions:
  ORG: 45
  PER: 31
  MISC: 27
  LOC: 18


In [13]:
# Build all_docs — combines train + test into one list with metadata
all_docs = []

for split_name, df in [('train', train_df), ('test', test_df)]:
    for idx, row in df.iterrows():
        tokens = list(row['tokens'])
        gold_tags = list(row['ner_tags'])
        text = tokens_to_text(tokens)
        gold_entities = get_gold_entities(row, label_list)
        all_docs.append({
            'split': split_name,
            'text': text,
            'tokens': tokens,
            'gold_tags': gold_tags,
            'gold_entities': gold_entities
        })

print(f'Total docs: {len(all_docs)}')
print(f'Train: {sum(1 for d in all_docs if d["split"]=="train")}')
print(f'Test:  {sum(1 for d in all_docs if d["split"]=="test")}')


Total docs: 6524
Train: 4836
Test:  1688


## 4. Run on Full Dataset & Save All Predictions

In [14]:
import json

def fix(obj):
    if isinstance(obj, dict):
        return {k: fix(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [fix(i) for i in obj]
    elif hasattr(obj, 'item'):  # numpy scalar
        return obj.item()
    else:
        return obj

with open('twitter_ner_predictions.json', 'w') as f:
    json.dump(fix(all_docs), f, indent=2)
print("Saved: twitter_ner_predictions.json")

Saved: twitter_ner_predictions.json


## 5. Export Batch 1 (10 docs) for Label Studio

In [15]:
batch1 = all_docs[:10]

label_studio_import = [{'text': doc['text']} for doc in batch1]
with open('batch1_label_studio.json', 'w') as f:
    json.dump(label_studio_import, f, indent=2)

print("Saved: batch1_label_studio.json — import this into Label Studio")
print("\nBatch 1 tweets (both annotators label these independently):")
for i, doc in enumerate(batch1):
    gold = [(e['text'], e['label']) for e in doc['gold_entities']]
    print(f"  [{i+1}] {doc['text']}")
    print(f"       Gold: {gold}")

Saved: batch1_label_studio.json — import this into Label Studio

Batch 1 tweets (both annotators label these independently):
  [1] I hate the words chunder , vomit and puke . BUUH .
       Gold: []
  [2] Alesan kenapa mlm kita lbh srg galau Poconggg '"' TwitFAKTA : Otak lebih aktif di malam hari dari pada di pagi hari . # TwitFAKTA '"'
       Gold: []
  [3] Complete Tosca on the tube http://t.co/O90deSLB
       Gold: []
  [4] Think you call that smash and grab . # Gateshead 's media man just admitted to me it was '"' daylight robbery '"' . Shaw 's only touch was his goal .
       Gold: [('Gateshead', 'ORG'), ('Shaw', 'PER')]
  [5] Happy New Year world !
       Gold: []
  [6] middle aged man band playing blink 182 . l0 l .
       Gold: [('blink 182 .', 'LOC')]
  [7] 14 days till my birthday : )
       Gold: []
  [8] Happy New Year ! It 's time to start thinking where you 'd like to visit in # 2012 . Need some ideas ? http://t.co/NpsUeTAG
       Gold: []
  [9] i 've got dressed but only 

## 5b. Export Batch 2 for Label Studio

Batch 2 layout (after filtering)

In [22]:
# Batch 2 — export for Label Studio annotation
# Uses cleaned all_docs (non-English and emoji tweets already removed)

batch2_s   = all_docs[10:110]   
batch2_a   = all_docs[110:210]  
batch2_overlap = all_docs[20:40]    # Both annotate docs 21-40 (overlap for IAA)

def save_label_studio_batch(docs, filename):
    import json
    items = [{'text': doc['text']} for doc in docs]
    with open(filename, 'w') as f:
        json.dump(items, f, indent=2)
    print(f"Saved {filename} ({len(items)} docs)")

save_label_studio_batch(batch2_sadaf,   'batch2a_sadaf_label_studio.json')
save_label_studio_batch(batch2_aziza,   'batch2b_aziza_label_studio.json')
save_label_studio_batch(batch2_overlap, 'batch2_overlap_label_studio.json')

print("\nBatch 2 summary:")
print(f"  Aziza (docs 11-110):   {len(batch2_sadaf)} docs")
print(f"  Sadaf (docs 111-210):  {len(batch2_aziza)} docs")
print(f"  Overlap (docs 21-40):  {len(batch2_overlap)} docs")

print("\nFirst 3 tweets in Aziza batch:")
for i, doc in enumerate(batch2_sadaf[:3]):
    print(f"  [{i+11}] {doc['text']}")

print("\nFirst 3 tweets in Sadaf batch:")
for i, doc in enumerate(batch2_aziza[:3]):
    print(f"  [{i+111}] {doc['text']}")

Saved batch2a_sadaf_label_studio.json (100 docs)
Saved batch2b_aziza_label_studio.json (100 docs)
Saved batch2_overlap_label_studio.json (20 docs)

Batch 2 summary:
  Aziza (docs 11-110):   100 docs
  Sadaf (docs 111-210):  100 docs
  Overlap (docs 21-40):  20 docs

First 3 tweets in Aziza batch:
  [11] Happy New Year to you all . Feeling pretty virtuous after organising a row for some mates and myself at 10 this morning . It was great . . . ish !
  [12] My dad thinks I ca n't hear him singing in the kitchen . . .
  [13] How did Dorothy Gale come back * younger * in Return to Oz ? ?

First 3 tweets in Sadaf batch:
  [111] That 229 was behind me , now its like a mile infront / :
  [112] Over the moon for a marra at work . He 's just been hooked up with a 6figure tax free job . This bodes well for my future in the business
  [113] Rediscovered this filmed , directed and produced by my cousin , starring another cousin . http://t.co/rFILlhKD # musicvideos # 50 fwd


## 6. IAA Calculation — Cohen's Kappa

In [17]:
from sklearn.metrics import cohen_kappa_score, f1_score, classification_report

# Annotator A — Sadaf:
annotator_a_labels = ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-PER", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-PROD", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-LOC", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-PER", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]

# Annotator B — Aziza:
annotator_b_labels = ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-PER", "I-PER", "O", "O", "O", "O", "O", "B-ORG", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-LOC", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-PER", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "B-ORG", "I-ORG", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"]

# ================================================================
# DO NOT EDIT BELOW THIS LINE
# ================================================================
assert len(annotator_a_labels) > 0, "Fill in annotator_a_labels first!"
assert len(annotator_b_labels) > 0, "Aziza needs to fill in annotator_b_labels!"
assert len(annotator_a_labels) == len(annotator_b_labels), f"Lists must be same length! A={len(annotator_a_labels)}, B={len(annotator_b_labels)}"

a_binary = [0 if l == "O" else 1 for l in annotator_a_labels]
b_binary = [0 if l == "O" else 1 for l in annotator_b_labels]

kappa = cohen_kappa_score(a_binary, b_binary)
f1    = f1_score(a_binary, b_binary)

print(f"Cohen's Kappa: {kappa:.3f}")
print(f"Entity F1:     {f1:.3f}")

if kappa >= 0.80:
    print("Status: Excellent — proceed to bulk annotation!")
elif kappa >= 0.70:
    print("Status: Acceptable — minor guideline tweaks needed.")
elif kappa >= 0.60:
    print("Status: Moderate — revise guidelines and re-annotate.")
else:
    print("Status: Poor — major guideline revision needed.")


Cohen's Kappa: 0.760
Entity F1:     0.769
Status: Acceptable — minor guideline tweaks needed.


In [18]:
# Per-entity breakdown — find where disagreements are happening
all_labels    = sorted(set(annotator_a_labels + annotator_b_labels))
entity_labels = [l for l in all_labels if l != 'O']

if entity_labels:
    print("Per-label agreement:")
    print(classification_report(annotator_a_labels, annotator_b_labels, labels=entity_labels))

Per-label agreement:
              precision    recall  f1-score   support

       B-LOC       1.00      1.00      1.00         1
       B-ORG       0.50      1.00      0.67         1
       B-PER       0.50      0.50      0.50         2
      B-PROD       0.00      0.00      0.00         1
       I-ORG       1.00      1.00      1.00         1
       I-PER       0.00      0.00      0.00         0

   micro avg       0.57      0.67      0.62         6
   macro avg       0.50      0.58      0.53         6
weighted avg       0.58      0.67      0.61         6



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/s

## 7. Export to CoNLL Format (for Model Training)

In [19]:
def export_to_conll(docs, output_file, label_list):
    with open(output_file, 'w') as f:
        for doc in docs:
            for token, tag_id in zip(doc['tokens'], doc['gold_tags']):
                label = label_list[tag_id]
                f.write(f"{token}\t{label}\n")
            f.write("\n")
    print(f"Saved: {output_file}")

train_docs = [d for d in all_docs if d['split'] == 'train']
test_docs  = [d for d in all_docs if d['split'] == 'test']

split_idx   = int(len(train_docs) * 0.8)
train_split = train_docs[:split_idx]
val_split   = train_docs[split_idx:]

export_to_conll(train_split, 'twitter_train.conll', label_list)
export_to_conll(val_split,   'twitter_val.conll',   label_list)
export_to_conll(test_docs,   'twitter_test.conll',  label_list)

print(f"\nSplit sizes — Train: {len(train_split)}, Val: {len(val_split)}, Test: {len(test_docs)}")

Saved: twitter_train.conll
Saved: twitter_val.conll
Saved: twitter_test.conll

Split sizes — Train: 3868, Val: 968, Test: 1688
